In [1]:
# ==========================================================
# CELL 1 — IMPORTS AND PATHS
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SQL_DIR = BASE_DIR / "sql"

REPORTS_DIR = BASE_DIR / "reports"
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", BASE_DIR)
print("Processed:", PROCESSED_DIR)
print("Reports:", REPORTS_DIR)

Project: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics
Processed: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed
Reports: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\reports


In [2]:
# ==========================================================
# CELL 2 — LOAD FEATURE-ENGINEERED DATA
# ==========================================================

players = pd.read_csv(
    PROCESSED_DIR / "players_features.csv"
)

teams = pd.read_csv(
    PROCESSED_DIR / "teams_features.csv"
)

matches = pd.read_csv(
    PROCESSED_DIR / "matches_features.csv"
)

print("Players:", players.shape)
print("Teams:", teams.shape)
print("Matches:", matches.shape)

Players: (1248, 80)
Teams: (48, 137)
Matches: (104, 53)


In [4]:
# ==========================================================
# CELL 3 — VERIFY BUSINESS ANALYSIS COLUMNS
# ==========================================================

required_player_columns = [
    "player",
    "team",
    "position",
    "age",
    "minutes",
    "goals",
    "assists",
    "goals_per90",
    "assists_per90",
    "goal_contributions",
    "goal_contributions_per90",
    "shooting_efficiency",
    "shot_accuracy",
    "defensive_actions",
    "defensive_actions_per90",
    "performance_index"
]

required_team_columns = [
    "team",
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "goals_per_shot",
    "shot_accuracy",
    "defensive_actions",
    "team_performance_index"
]

required_match_columns = [
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "total_goals",
    "result",
    "possession_difference",
    "shot_difference",
    "shots_on_target_difference",
    "match_intensity"
]

print("Missing player columns:")
print([
    col for col in required_player_columns
    if col not in players.columns
])

print("\nMissing team columns:")
print([
    col for col in required_team_columns
    if col not in teams.columns
])

print("\nMissing match columns:")
print([
    col for col in required_match_columns
    if col not in matches.columns
])

Missing player columns:
[]

Missing team columns:
[]

Missing match columns:
[]


In [5]:
# ==========================================================
# CELL 4 — SCOUTING CANDIDATES
# ==========================================================

scouting_candidates = players[
    players["minutes"].fillna(0) >= 180
].copy()

print(
    "Players eligible for scouting analysis:",
    len(scouting_candidates)
)

Players eligible for scouting analysis: 532


In [6]:
# ==========================================================
# CELL 5 — TOP ATTACKING PLAYERS
# ==========================================================

top_attackers = (
    scouting_candidates
    .sort_values(
        [
            "goal_contributions_per90",
            "minutes"
        ],
        ascending=[False, False]
    )
    [
        [
            "player",
            "team",
            "position",
            "age",
            "minutes",
            "goals",
            "assists",
            "goal_contributions_per90",
            "performance_index"
        ]
    ]
    .head(20)
)

display(top_attackers)

,player,team,position,age,minutes,goals,assists,goal_contributions_per90,performance_index
1104,Johan Manzambi,Switzerland,"MF,FW",20,199.0,3.0,2.0,2.26,1.356
477,Kylian Mbappé,France,FW,27,695.0,10.0,4.0,1.81,1.138
123,Romelu Lukaku,Belgium,FW,33,233.0,3.0,1.0,1.55,0.930
443,Bukayo Saka,England,MF,24,358.0,3.0,3.0,1.50,1.700
226,Nathan Saliba,Canada,MF,22,182.0,1.0,2.0,1.48,2.488
41,Lionel Messi,Argentina,FW,38,740.0,8.0,4.0,1.46,1.364
807,Andreas Schjelderup,Norway,FW,22,252.0,1.0,3.0,1.43,2.001
757,Crysencio Summerville,Netherlands,FW,24,253.0,2.0,2.0,1.42,1.424
811,Erling Haaland,Norway,FW,25,465.0,7.0,0.0,1.35,0.887
1000,Ismaila Sarr,Senegal,"MF,FW",28,364.0,4.0,1.0,1.24,1.044


In [7]:
# ==========================================================
# CELL 6 — YOUNG HIGH-POTENTIAL PLAYERS
# ==========================================================

young_players = scouting_candidates[
    scouting_candidates["age"] <= 23
].copy()

young_high_potential = (
    young_players
    .sort_values(
        "performance_index",
        ascending=False
    )
    [
        [
            "player",
            "team",
            "position",
            "age",
            "minutes",
            "goals",
            "assists",
            "goal_contributions_per90",
            "performance_index"
        ]
    ]
    .head(20)
)

display(young_high_potential)

,player,team,position,age,minutes,goals,assists,goal_contributions_per90,performance_index
226,Nathan Saliba,Canada,MF,22,182.0,1.0,2.0,1.48,2.488
328,Livano Comenencia,Curaçao,MF,22,233.0,1.0,0.0,0.39,2.080
807,Andreas Schjelderup,Norway,FW,22,252.0,1.0,3.0,1.43,2.001
863,Diego Gómez,Paraguay,MF,23,304.0,0.0,0.0,0.00,1.765
368,Christ Inao Oulaï,Côte d'Ivoire,MF,20,208.0,0.0,0.0,0.00,1.739
365,Amad Diallo,Côte d'Ivoire,"MF,FW",23,185.0,2.0,0.0,0.97,1.725
449,Elliot Anderson,England,MF,23,632.0,0.0,1.0,0.14,1.570
76,Paul Okon-Engstler,Australia,MF,21,197.0,0.0,1.0,0.46,1.549
1171,Alex Freeman,United States,DF,21,374.0,1.0,1.0,0.48,1.526
458,Jude Bellingham,England,MF,22,614.0,7.0,1.0,1.18,1.473


In [8]:
# ==========================================================
# CELL 7 — DEFENSIVE SCOUTING
# ==========================================================

defensive_candidates = (
    scouting_candidates[
        scouting_candidates["position"].str.contains(
            "DF",
            na=False
        )
    ]
    .sort_values(
        "defensive_actions_per90",
        ascending=False
    )
    [
        [
            "player",
            "team",
            "position",
            "age",
            "minutes",
            "tackles_won",
            "interceptions",
            "defensive_actions_per90",
            "performance_index"
        ]
    ]
    .head(20)
)

display(defensive_candidates)

,player,team,position,age,minutes,tackles_won,interceptions,defensive_actions_per90,performance_index
543,Marvin Senaya,Ghana,DF,25,278.0,12.0,6.0,5.806,2.322
21,Rayan Aït-Nouri,Algeria,DF,25,284.0,13.0,5.0,5.625,2.250
616,Merchas Doski,Iraq,DF,26,270.0,9.0,6.0,5.000,2.000
1021,Khuliso Mudau,South Africa,DF,31,360.0,8.0,11.0,4.750,1.900
1132,Mohamed Amine Ben Hamida,Tunisia,DF,30,202.0,9.0,1.0,4.545,1.818
1119,Ali Abdi,Tunisia,DF,32,269.0,8.0,4.0,4.000,1.600
674,Yazan Al-Arab,Jordan,DF,30,270.0,1.0,11.0,4.000,1.600
102,Stefan Posch,Austria,DF,29,331.0,4.0,10.0,3.784,1.514
472,Dayot Upamecano,France,DF,27,660.0,12.0,15.0,3.699,1.564
959,Saud Abdulhamid,Saudi Arabia,DF,26,269.0,9.0,2.0,3.667,1.467


In [9]:
# ==========================================================
# CELL 8 — TEAM ATTACKING PERFORMANCE
# ==========================================================

team_attack_analysis = (
    teams
    [
        [
            "team",
            "games",
            "goals",
            "assists",
            "shots",
            "shots_on_target",
            "goals_per90",
            "goals_per_shot",
            "shot_accuracy"
        ]
    ]
    .sort_values(
        "goals_per90",
        ascending=False
    )
)

display(
    team_attack_analysis.head(20)
)

,team,games,goals,assists,shots,shots_on_target,goals_per90,goals_per_shot,shot_accuracy
19,Germany,4,11,10,74,28,2.54,0.1486,0.3784
18,France,8,20,18,139,59,2.50,0.1439,0.4245
17,England,8,20,14,118,53,2.40,0.1695,0.4492
29,Netherlands,4,10,10,46,22,2.31,0.2174,0.4783
38,Senegal,4,10,9,69,23,2.31,0.1449,0.3333
4,Belgium,6,13,10,112,34,2.05,0.1161,0.3036
6,Brazil,5,10,8,74,30,2.00,0.1351,0.4054
1,Argentina,8,18,12,114,44,2.00,0.1579,0.3860
24,Japan,4,8,7,34,13,2.00,0.2353,0.3824
27,Mexico,5,10,7,70,21,2.00,0.1429,0.3000


In [10]:
# ==========================================================
# CELL 9 — TEAM EFFICIENCY
# ==========================================================

team_efficiency = (
    teams[
        [
            "team",
            "goals",
            "shots",
            "shots_on_target",
            "goals_per_shot",
            "shot_accuracy",
            "team_performance_index"
        ]
    ]
    .sort_values(
        "goals_per_shot",
        ascending=False
    )
)

display(
    team_efficiency.head(20)
)

,team,goals,shots,shots_on_target,goals_per_shot,shot_accuracy,team_performance_index
24,Japan,8,34,13,0.2353,0.3824,1.375
29,Netherlands,10,46,22,0.2174,0.4783,1.845
31,Norway,12,66,29,0.1818,0.4394,1.105
17,England,20,118,53,0.1695,0.4492,1.680
11,Croatia,6,37,17,0.1622,0.4595,0.625
1,Argentina,18,114,44,0.1579,0.3860,1.610
3,Austria,5,32,8,0.1562,0.2500,0.250
45,United States,9,59,19,0.1525,0.3220,1.200
19,Germany,11,74,28,0.1486,0.3784,1.960
41,Sweden,7,48,23,0.1458,0.4792,0.500


In [11]:
# ==========================================================
# CELL 10 — TEAM DEFENSIVE PERFORMANCE
# ==========================================================

team_defensive_analysis = (
    teams[
        [
            "team",
            "defensive_actions",
            "tackles_won",
            "interceptions",
            "plus_minus_per90",
            "team_performance_index"
        ]
    ]
    .sort_values(
        "defensive_actions",
        ascending=False
    )
)

display(
    team_defensive_analysis.head(20)
)

,team,defensive_actions,tackles_won,interceptions,plus_minus_per90,team_performance_index
1,Argentina,175,93,82,1.22,1.610
40,Spain,152,88,64,1.56,1.560
18,France,142,77,65,1.25,1.875
33,Paraguay,141,88,53,-0.56,0.000
17,England,128,77,51,0.96,1.680
31,Norway,111,67,44,0.32,1.105
42,Switzerland,104,52,52,0.60,1.050
45,United States,102,42,60,0.60,1.200
28,Morocco,101,61,40,0.63,1.105
16,Egypt,98,54,44,0.19,0.845


In [12]:
# ==========================================================
# CELL 11 — MATCH OUTCOME ANALYSIS
# ==========================================================

match_outcomes = (
    matches["result"]
    .value_counts()
    .rename_axis("result")
    .reset_index(name="matches")
)

match_outcomes["percentage"] = (
    match_outcomes["matches"]
    /
    len(matches)
    *
    100
).round(2)

display(match_outcomes)

,result,matches,percentage
0,Home Win,50,48.08
1,Away Win,30,28.85
2,Draw,24,23.08


In [13]:
# ==========================================================
# CELL 12 — MATCH RESULT DRIVERS
# ==========================================================

match_drivers = (
    matches
    .groupby("result")
    [
        [
            "total_goals",
            "possession_difference",
            "shot_difference",
            "shots_on_target_difference",
            "match_intensity"
        ]
    ]
    .mean()
    .round(2)
)

display(match_drivers)

,total_goals,possession_difference,shot_difference,shots_on_target_difference,match_intensity
result,,,,,
Away Win,3.13,-13.20,-2.60,-2.60,47.20
Draw,1.50,11.92,2.42,0.33,49.71
Home Win,3.44,12.02,7.16,3.84,47.62


In [14]:
# ==========================================================
# CELL 13 — HIGHEST-SCORING MATCHES
# ==========================================================

highest_scoring_matches = (
    matches[
        [
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "total_goals",
            "result"
        ]
    ]
    .sort_values(
        "total_goals",
        ascending=False
    )
    .head(15)
)

display(highest_scoring_matches)

,home_team,away_team,home_score,away_score,total_goals,result
102,France,England,4.0,6.0,10.0,Away Win
8,Germany,Curaçao,7.0,1.0,8.0,Home Win
11,Sweden,Tunisia,5.0,1.0,6.0,Home Win
26,Canada,Qatar,6.0,0.0,6.0,Home Win
32,Netherlands,Sweden,5.0,1.0,6.0,Home Win
65,New Zealand,Belgium,1.0,5.0,6.0,Away Win
51,Morocco,Haiti,4.0,2.0,6.0,Home Win
21,England,Croatia,4.0,2.0,6.0,Home Win
71,Algeria,Austria,3.0,3.0,6.0,Draw
44,Portugal,Uzbekistan,5.0,0.0,5.0,Home Win


In [15]:
# ==========================================================
# CELL 14 — TEAM OPPORTUNITY ANALYSIS
# ==========================================================

team_opportunity = teams[
    [
        "team",
        "goals",
        "shots",
        "shots_on_target",
        "goals_per_shot",
        "shot_accuracy"
    ]
].copy()

team_opportunity["shots_per_goal"] = np.where(
    team_opportunity["goals"] > 0,
    team_opportunity["shots"]
    /
    team_opportunity["goals"],
    np.nan
)

team_opportunity = (
    team_opportunity
    .sort_values(
        "shots_per_goal",
        ascending=False
    )
)

display(
    team_opportunity.head(20)
)

,team,goals,shots,shots_on_target,goals_per_shot,shot_accuracy,shots_per_goal
21,Haiti,1,31,7,0.0323,0.2258,31.000000
37,Scotland,1,29,7,0.0345,0.2414,29.000000
12,Curaçao,1,29,7,0.0345,0.2414,29.000000
15,Ecuador,2,53,20,0.0377,0.3774,26.500000
44,Türkiye,3,71,16,0.0423,0.2254,23.666667
2,Australia,2,42,12,0.0476,0.2857,21.000000
23,Iraq,1,21,2,0.0476,0.0952,21.000000
39,South Africa,2,39,11,0.0513,0.2821,19.500000
9,Colombia,5,94,30,0.0532,0.3191,18.800000
13,Czechia,2,34,8,0.0588,0.2353,17.000000


In [16]:
# ==========================================================
# CELL 15 — RECRUITMENT SHORTLIST
# ==========================================================

recruitment = scouting_candidates.copy()

recruitment["recruitment_score"] = (
    recruitment["performance_index"] * 0.7
    +
    recruitment["goal_contributions_per90"] * 0.3
)

recruitment["recruitment_score"] = (
    recruitment["recruitment_score"]
    .round(3)
)

recruitment_shortlist = (
    recruitment
    .sort_values(
        "recruitment_score",
        ascending=False
    )
    [
        [
            "player",
            "team",
            "position",
            "age",
            "minutes",
            "goal_contributions_per90",
            "defensive_actions_per90",
            "performance_index",
            "recruitment_score"
        ]
    ]
    .head(25)
)

display(recruitment_shortlist)

,player,team,position,age,minutes,goal_contributions_per90,defensive_actions_per90,performance_index,recruitment_score
226,Nathan Saliba,Canada,MF,22,182.0,1.48,4.000,2.488,2.186
807,Andreas Schjelderup,Norway,FW,22,252.0,1.43,2.857,2.001,1.830
443,Bukayo Saka,England,MF,24,358.0,1.50,2.000,1.700,1.640
1104,Johan Manzambi,Switzerland,"MF,FW",20,199.0,2.26,0.000,1.356,1.627
543,Marvin Senaya,Ghana,DF,25,278.0,0.00,5.806,2.322,1.625
469,Aurélien Tchouaméni,France,MF,26,360.0,0.25,5.000,2.150,1.580
21,Rayan Aït-Nouri,Algeria,DF,25,284.0,0.00,5.625,2.250,1.575
328,Livano Comenencia,Curaçao,MF,22,233.0,0.39,4.615,2.080,1.573
877,Matías Galarza,Paraguay,MF,24,388.0,0.46,4.186,1.950,1.503
501,Felix Nmecha,Germany,MF,25,270.0,0.66,3.667,1.863,1.502


In [17]:
# ==========================================================
# CELL 16 — BUSINESS KPI SUMMARY
# ==========================================================

business_kpis = pd.DataFrame({
    "metric": [
        "Total Players",
        "Total Teams",
        "Total Matches",
        "Total Goals",
        "Average Goals Per Match",
        "Average Match Intensity",
        "Top Scorer Goals",
        "Highest Team Goals"
    ],
    "value": [
        len(players),
        len(teams),
        len(matches),
        matches["total_goals"].sum(),
        matches["total_goals"].mean(),
        matches["match_intensity"].mean(),
        players["goals"].max(),
        teams["goals"].max()
    ]
})

display(business_kpis)

,metric,value
0,Total Players,1248.000000
1,Total Teams,48.000000
2,Total Matches,104.000000
3,Total Goals,302.000000
4,Average Goals Per Match,2.903846
5,Average Match Intensity,47.980769
6,Top Scorer Goals,10.000000
7,Highest Team Goals,20.000000


In [18]:
# ==========================================================
# CELL 17 — SAVE BUSINESS ANALYSIS OUTPUTS
# ==========================================================

outputs = {
    "top_attackers": top_attackers,
    "young_high_potential": young_high_potential,
    "defensive_candidates": defensive_candidates,
    "team_attack_analysis": team_attack_analysis,
    "team_efficiency": team_efficiency,
    "team_defensive_analysis": team_defensive_analysis,
    "match_outcomes": match_outcomes,
    "match_drivers": match_drivers.reset_index(),
    "highest_scoring_matches": highest_scoring_matches,
    "team_opportunity": team_opportunity,
    "recruitment_shortlist": recruitment_shortlist,
    "business_kpis": business_kpis
}

for name, df in outputs.items():

    df.to_csv(
        REPORTS_DIR / f"{name}.csv",
        index=False
    )

    print(
        f"Saved: reports/{name}.csv"
    )

Saved: reports/top_attackers.csv
Saved: reports/young_high_potential.csv
Saved: reports/defensive_candidates.csv
Saved: reports/team_attack_analysis.csv
Saved: reports/team_efficiency.csv
Saved: reports/team_defensive_analysis.csv
Saved: reports/match_outcomes.csv
Saved: reports/match_drivers.csv
Saved: reports/highest_scoring_matches.csv
Saved: reports/team_opportunity.csv
Saved: reports/recruitment_shortlist.csv
Saved: reports/business_kpis.csv


In [19]:
# ==========================================================
# CELL 18 — FINAL VALIDATION
# ==========================================================

print("=" * 70)
print("BUSINESS ANALYSIS COMPLETE")
print("=" * 70)

print("\nKey outputs:")

for name in outputs:
    print(f"✓ {name}.csv")

print("\nReports directory:")
print(REPORTS_DIR)

BUSINESS ANALYSIS COMPLETE

Key outputs:
✓ top_attackers.csv
✓ young_high_potential.csv
✓ defensive_candidates.csv
✓ team_attack_analysis.csv
✓ team_efficiency.csv
✓ team_defensive_analysis.csv
✓ match_outcomes.csv
✓ match_drivers.csv
✓ highest_scoring_matches.csv
✓ team_opportunity.csv
✓ recruitment_shortlist.csv
✓ business_kpis.csv

Reports directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\reports
